# Module 6: Chat LangChain — Workflow vs Deep Agent, End-to-End ADLC

> Part of the **Modular Workshops** series. Standalone, ~50 min.
> Needs: a LangSmith **service key** (`lsv2_sk_...`) to deploy, an `OPENAI_API_KEY` **workspace secret** in LangSmith (Settings → Secrets) for the online judges and Insights, and a Plus/Enterprise plan for Part 6 (everything else runs without it).

Modules 1–5 each teach one slice of the Agent Development Lifecycle. This module runs **one agent through the whole loop** — and does it twice, so we can compare a hand-wired **LangGraph workflow** against a **Deep Agent** on the same job: chatting with the LangChain docs through the public [LangChain docs MCP servers](https://docs.langchain.com/use-these-docs).

Six parts:

1. **Build** — two agents, one task. A deterministic RAG workflow and an agentic Deep Agent, both on the docs + API-reference MCP servers.
2. **Deploy** — one LangSmith deployment serving both graphs. Invoke them from Studio, from a chat page packaged *in* the server, or from Agent Chat UI.
3. **Online evaluators** — four scenarios, all set up from the SDK: a run-level LLM judge, a code evaluator, a thread-level (multi-turn) judge, and a chained evaluator that reads other evaluators' scores.
4. **Automation → annotation queue** — route flagged runs to a human, who writes *assertions* rather than golden answers.
5. **Offline evaluation** — a claim-based dataset, both agents scored against it, a pairwise judge, and a cost/latency table.
6. **Insights** — an Insights report created and scheduled from code.

<img src="../images/adlc.avif" style="width: auto; max-height: 360px; border-radius: 8px;">

## Setup

In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(dotenv_path=project_root / ".env", override=True)

import os, re, time, json
from collections import defaultdict
from datetime import datetime, timedelta, timezone
from typing import Literal
from pydantic import BaseModel, Field

from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langsmith import Client, aevaluate, evaluate, uuid7

from utils.models import model
from utils.utils import show_graph
from utils.engine import deployment_url
from utils.langsmith_rules import (
    create_run_rule, delete_run_rule, list_run_rules, set_annotation_queue_default_dataset,
)
from utils.langsmith_evaluators import (
    push_judge_prompt, upsert_evaluator, attach_evaluator, run_code_evaluator_locally, run_to_dict,
)
from utils.langsmith_insights import create_insights_report, save_insights_config, print_insights_summary
from utils.langsmith_model_configs import print_model_configurations

client = Client()
DEPLOYMENT_NAME = "modular-workshops-chat-langchain"   # the deployment's tracing project has the same name

print("LANGSMITH_API_KEY set:", bool(os.environ.get("LANGSMITH_API_KEY")))
print("OPENAI_API_KEY set:   ", bool(os.environ.get("OPENAI_API_KEY")))
print("Deployment / project: ", DEPLOYMENT_NAME)

---
# Part 1. Build — two agents, one task

Both agents answer questions about LangChain, LangGraph, Deep Agents, and LangSmith, and both get their knowledge the same way: the two **public MCP servers** that LangChain publishes for its docs. No API key, streamable-HTTP transport, and `langchain-mcp-adapters` turns each server's tools into ordinary LangChain tools.

| Server | URL | Tools we use |
|---|---|---|
| `docs` | `https://docs.langchain.com/mcp` | `search_docs_by_lang_chain` (semantic search), `query_docs_filesystem_docs_by_lang_chain` (read a page) |
| `reference` | `https://reference.langchain.com/mcp` | `search_api` (find a symbol), `get_symbol` (full signature + parameters) |

The shared loader lives in `agents/chat_langchain/mcp_tools.py`. It caches the tool list after the first call and **drops `submit_feedback`** — that tool files a report with the docs team, which is not something an agent should do on a user's behalf.

> MCP tools are async-only. That is why everything in this module is `await`ed and why we use `aevaluate` later instead of Module 4's `evaluate`.

### 1.1 The MCP tools

In [ ]:
from agents.chat_langchain.mcp_tools import get_mcp_tools, DOCS_TOOLS, REFERENCE_TOOLS

tools = await get_mcp_tools()
for name, tool in tools.items():
    server = "docs" if name in DOCS_TOOLS else "reference"
    print(f"[{server:9}] {name}({', '.join(tool.args)})")
    print(f"            {tool.description.strip().splitlines()[0][:110]}")

### 1.2 Agent A — a LangGraph RAG workflow

`agents/chat_langchain/rag_workflow.py` is the Module 2 style: **you** decide the control flow, the model fills in the blanks.

- `plan_query` — structured output: rewrite the latest message into a standalone search query and pick a **route** (`docs`, `reference`, or `both`).
- `retrieve_docs` / `retrieve_reference` — call the MCP tools directly. Every call is a tool run in the trace.
- `generate` — answer **only** from the retrieved context, citing URLs.

The state extends `MessagesState` and adds `query`, `route`, `context`, `sources`, and `tool_calls`. Keeping `messages` as the contract matters later: Studio's chat mode, the packaged UI, thread-level evaluators, and one shared eval dataset all rely on it.

In [ ]:
from agents.chat_langchain.rag_workflow import build_graph

rag = build_graph(checkpointer=MemorySaver())   # the deployment gets its own checkpointer
show_graph(rag)

In [ ]:
question = "How do I add middleware to an agent built with create_agent?"

rag_config = {"configurable": {"thread_id": str(uuid7())}}
rag_result = await rag.ainvoke({"messages": [{"role": "user", "content": question}]}, config=rag_config)

print(rag_result["messages"][-1].text)
print("\nroute:      ", rag_result["route"])
print("tool calls: ", rag_result["tool_calls"])
print("sources:    ", *rag_result["sources"][:3], sep="\n  ")

### 1.3 Agent B — a Deep Agent

`agents/chat_langchain/deep_agent.py` is the Module 1 style: give the model the tools and a good `AGENTS.md`, and let it plan. Two differences from the workflow that matter for the comparison:

- It also gets the docs **filesystem** tool, so it can read a whole page when a search hit is truncated.
- Exact API questions are delegated to an `api-reference` **subagent** that owns the reference-server tools (`search_api`, `get_symbol`).

The factory is async because the tools have to be loaded before `create_deep_agent` can bind them.

In [ ]:
from agents.chat_langchain.deep_agent import make_local_agent, SYSTEM_PROMPT

deep = await make_local_agent(checkpointer=MemorySaver())
print(SYSTEM_PROMPT)

In [ ]:
def extract_tool_calls(messages) -> list[str]:
    """Tool-call names in order, from an agent's message list."""
    return [tc["name"] for m in messages for tc in (getattr(m, "tool_calls", None) or [])]

deep_config = {"configurable": {"thread_id": str(uuid7())}}
deep_result = await deep.ainvoke({"messages": [{"role": "user", "content": question}]}, config=deep_config)

print(deep_result["messages"][-1].text)
print("\ntool calls:", extract_tool_calls(deep_result["messages"]))

### 1.4 Same question, two very different traces

Open the two traces in LangSmith and compare them side by side. Before any numbers, the shape of the difference:

| | LangGraph workflow (`rag_workflow`) | Deep Agent (`chat_deep_agent`) |
|---|---|---|
| Who decides to search | You — every turn retrieves | The model — 0..N searches, may read full pages |
| Steps per turn | Fixed: 3–4 nodes, 1–2 tool calls | Variable: planning, subagent delegation, re-search |
| Latency / cost | Predictable, low | Higher and variable |
| Multi-hop / API-precise questions | Only as good as one search | Can drill into a symbol, cross-check pages |
| What you can evaluate | Each node's output (query, route, context) | Final answer + trajectory |
| Failure mode | Bad retrieval → confident "not in the docs" | Over-searching, long answers, occasional wandering |

Part 5 puts numbers on this table.

---
# Part 2. Deploy once, invoke three ways

Both graphs ship from **one** deployment described by `langgraph.chat_langchain.json` (separate from Module 3's `langgraph.json`, so that deployment is untouched). One deployment means one tracing project — every online evaluator, automation, and Insights report in the rest of this module is configured **once** and applies to both agents. The traces stay distinguishable through the `graph_id` metadata the server stamps on every run.

The config also registers an `http.app`: a FastAPI app whose only route serves a single-file chat page. It's the "lightweight UI packaged in the agent server" — it calls the server's own `/threads` and `/runs/stream` endpoints on the same origin.

### 2.1 The config, validated

In [ ]:
print((project_root / "langgraph.chat_langchain.json").read_text())
print((project_root / "agents" / "chat_langchain" / "webapp.py").read_text())

In [ ]:
# `validate` imports each graph and checks the config without building anything.
!cd "{project_root}" && langgraph validate --config langgraph.chat_langchain.json

### 2.2 Deploy to LangSmith (optional — presenters: run this before the session)

Same caveats as Module 3 §2.5: needs a service key, builds locally if Docker is available (otherwise remotely), takes a few minutes to provision. Re-running pushes a new revision to the same deployment.

To iterate locally instead: `langgraph dev --config langgraph.chat_langchain.json` gives you Studio and `http://127.0.0.1:2024/chat`; set `LANGSMITH_DEPLOYMENT_URL=http://127.0.0.1:2024` and the cells below work against it (traces then land in your `LANGSMITH_PROJECT` instead of the deployment's project).

In [ ]:
!cd "{project_root}" && langgraph deploy --config langgraph.chat_langchain.json --name {DEPLOYMENT_NAME} --no-input

### 2.3 Three ways in

1. **Studio** — the built-in debugger. Graph mode shows every node; chat mode is a plain conversation with a dropdown to switch between `rag_workflow` and `chat_deep_agent`.
2. **The packaged chat page** — `/chat` on the deployment. Paste your LangSmith API key once (stored in the browser), pick an agent, chat. Every message is a run in the deployment's tracing project, in a thread.
3. **[Agent Chat UI](https://agentchat.vercel.app)** — the hosted open-source UI: enter the deployment URL, a graph id, and your key.

**Docs:** [Studio](https://docs.langchain.com/langsmith/use-studio) · [Custom routes](https://docs.langchain.com/langsmith/custom-routes) · [Agent Chat UI](https://docs.langchain.com/oss/python/langchain/ui)

In [ ]:
from langgraph_sdk import get_sync_client

DEPLOYMENT_URL = deployment_url(DEPLOYMENT_NAME)      # honours $LANGSMITH_DEPLOYMENT_URL if set
sdk = get_sync_client(url=DEPLOYMENT_URL)             # reads LANGSMITH_API_KEY from the environment

# Local `langgraph dev` traces go to $LANGSMITH_PROJECT; a cloud deployment traces to a project of its own name.
PROJECT_NAME = DEPLOYMENT_NAME if "langgraph.app" in DEPLOYMENT_URL else os.environ.get("LANGSMITH_PROJECT", "modular-workshops")

graphs = sorted({a["graph_id"] for a in sdk.assistants.search(limit=50)})
print("Deployment:", DEPLOYMENT_URL)
print("Graphs:    ", graphs)
print("Project:   ", PROJECT_NAME)
print(f"\n1. Studio:          https://smith.langchain.com/studio/?baseUrl={DEPLOYMENT_URL}")
print(f"2. Packaged UI:     {DEPLOYMENT_URL}/chat")
print(f"3. Agent Chat UI:   https://agentchat.vercel.app   (deployment URL above + graph id + your LangSmith key)")

### 2.4 Seed some production-shaped traffic

Everything downstream — thread-level judges, the annotation queue, Insights — needs real conversations, so we script a handful of **multi-turn threads** and alternate the two agents. The last two conversations include a vague follow-up and a user correction on purpose: those are the traces the thread-level judge should catch.

Attendees: chat through the packaged page or Studio at the same time; it all lands in the same project.

In [ ]:
conversations = [
    ("rag_workflow",    ["How do I add middleware to an agent built with create_agent?",
                         "Can you show a before_model example?"]),
    ("chat_deep_agent", ["How do I add middleware to an agent built with create_agent?",
                         "Can you show a before_model example?"]),
    ("rag_workflow",    ["What does init_chat_model return, and what are its parameters?",
                         "Does it support the OpenAI Responses API?"]),
    ("chat_deep_agent", ["What does init_chat_model return, and what are its parameters?",
                         "Does it support the OpenAI Responses API?"]),
    ("rag_workflow",    ["How do I pause a LangGraph graph for human approval?",
                         "and how do I resume it?",
                         "what if I want to edit the tool arguments before resuming?"]),
    ("chat_deep_agent", ["How do I connect a Deep Agent to an MCP server?",
                         "what about stdio servers?",
                         "no, that's not what I asked — I meant over HTTP with an auth header"]),
]

def message_text(message: dict) -> str:
    content = message.get("content", "")
    return content if isinstance(content, str) else "".join(b.get("text", "") for b in content if isinstance(b, dict))

def run_conversation(graph_id: str, turns: list[str]) -> str:
    thread = sdk.threads.create()
    for turn in turns:
        t0 = time.perf_counter()
        state = sdk.runs.wait(thread["thread_id"], graph_id, input={"messages": [{"role": "user", "content": turn}]})
        if "messages" not in state:            # runs.wait sometimes returns only {"__interrupt__": []}
            state = sdk.threads.get(thread["thread_id"])["values"]
        print(f"[{graph_id:15}] {time.perf_counter() - t0:5.1f}s  {turn[:46]:<46} -> {message_text(state['messages'][-1])[:60]!r}")
    return thread["thread_id"]

thread_ids: dict[str, list[str]] = defaultdict(list)
for graph_id, turns in conversations:
    thread_ids[graph_id].append(run_conversation(graph_id, turns))

project = client.read_project(project_name=PROJECT_NAME)
print("\nTraces: ", project.url)
print("Threads:", f"{project.url}?runview=threads")

---
# Part 3. Online evaluators — four scenarios, all from code

Module 4 created one online evaluator by POSTing an inline prompt to the run-rules endpoint. Since `langsmith>=0.9.8` there is a cleaner split:

- **Evaluators are workspace resources** — `client.evaluators.create(...)` (LLM-as-judge or code). They appear in the **Evaluators** table and can be reused across projects and datasets.
- **Attaching one to a project is a rule** — `POST /runs/rules` with `evaluator_id`, a filter, a sampling rate, and (for multi-turn) `group_by="thread_id"`. The SDK has no method for that step yet, so `utils/langsmith_rules.py` wraps it (`attach_evaluator`).

Four scenarios, each demonstrating something different:

| # | Evaluator | Scope | Demonstrates |
|---|---|---|---|
| 3.1 | `answer_quality` | run-level, LLM judge | Structured prompt in the hub + `variable_mapping` from run fields |
| 3.2 | `cites_docs` | run-level, code | Sandboxed Python over the run dict; test locally first |
| 3.3 | `conversation_resolved` | **thread-level**, LLM judge | `group_by="thread_id"`, judged once per idle conversation |
| 3.4 | `needs_review` | run-level, code, **chained** | `include_extended_stats` to read other evaluators' scores |

All of them apply to **both** agents — that's the point of one project. Sampling rate and per-evaluator spend limits are the cost knobs; we leave sampling at 100% so results show up during the session.

**Docs:** [Manage evaluators with the SDK](https://docs.langchain.com/langsmith/manage-evaluators-sdk) · [Online code evaluators](https://docs.langchain.com/langsmith/online-evaluations-code) · [Multi-turn online evaluators](https://docs.langchain.com/langsmith/online-evaluations-multi-turn)

### 3.0 Pick model configurations by name (optional)

LLM judges and Insights don't have to run on a bare provider default. A **model configuration** (Settings → Model configurations) is a named bundle of provider, model, parameters, and the secret holding the key, and each one has per-feature switches in the *Feature Access* table. The helpers in this module take the configuration's **name** and resolve it for you: an evaluator gets `playground_settings_id`, Insights gets `cluster_model` (Thinking) and `summary_model` (Summarization). Leave a variable as `None` to keep the default — the model in the hub commit for judges, the provider's defaults for Insights.

**Docs:** [Manage model configurations](https://docs.langchain.com/langsmith/model-configurations)

In [ ]:
print_model_configurations(client)

# Names exactly as they appear in Settings → Model configurations. None = LangSmith's default.
JUDGE_MODEL_CONFIG = "gpt-5.5-custom-0"          # e.g. "eval-judge"           (needs 'Evaluators' enabled)
INSIGHTS_THINKING_CONFIG = "gpt-5.5-custom-0"    # e.g. "insights-thinking"    (needs 'Insights (Thinking)' enabled)
INSIGHTS_SUMMARY_CONFIG = "gpt-5.4-mini-custom"     # e.g. "insights-summarizer"  (needs 'Insights (Summarization)' enabled)

### 3.1 Run-level LLM-as-judge (`answer_quality`)

An SDK-created LLM evaluator references a **structured prompt** in the Prompt Hub. `push_judge_prompt` builds a `StructuredPrompt` from the messages + a Pydantic schema, pushes `prompt | judge_model` so the commit carries a default model, and returns the commit hash. If `JUDGE_MODEL_CONFIG` is set, that configuration wins over the hub model. LangSmith resolves the judge's API key from the workspace secret `OPENAI_API_KEY` at run time — nothing from your `.env` leaves this machine.

Each schema field becomes a feedback key; a `comment` string becomes the feedback comment. The judge is **reference-free** — it only needs the question and the response.

In [ ]:
class AnswerQualityGrade(BaseModel):
    """Reference-free grade for one assistant turn."""
    answer_quality: bool = Field(description=(
        "True only if the response directly answers the user's latest question, cites at least one "
        "https://docs.langchain.com or https://reference.langchain.com URL, and invents no APIs, parameters, or URLs."
    ))
    comment: str = Field(description="One sentence explaining the grade.")

ANSWER_QUALITY_SYSTEM = (
    "You grade one turn of an assistant that answers questions about LangChain, LangGraph, Deep Agents, and LangSmith "
    "from the official documentation. Judge only the assistant's latest response to the user's latest question."
)
ANSWER_QUALITY_HUMAN = (
    "Conversation so far (JSON messages):\n{question}\n\n"
    "Assistant response messages (JSON):\n{response}"
)

pushed = push_judge_prompt(
    client, "chat-langchain-answer-quality",
    system=ANSWER_QUALITY_SYSTEM, human=ANSWER_QUALITY_HUMAN, schema=AnswerQualityGrade,
)
print("Prompt commit:", pushed["url"])

answer_quality_id = await upsert_evaluator(
    client, "chat-langchain-answer-quality", type="llm",
    llm_evaluator={
        "prompt_repo_handle": pushed["handle"],
        "commit_hash_or_tag": pushed["commit"],
        # prompt variable -> field of the run being scored
        "variable_mapping": {"question": "inputs.messages", "response": "outputs.messages"},
    },
    model_configuration=JUDGE_MODEL_CONFIG,   # by name; None keeps the hub commit's model
)
rule = attach_evaluator(
    client, answer_quality_id,
    project_name=PROJECT_NAME, display_name="answer-quality (LLM judge)",
    filter="eq(is_root, true)",   # score the trace, not every child LLM/tool span
    sampling_rate=1.0,
)
print("Evaluator:", answer_quality_id)
print("Rule:     ", rule["url"])

### 3.2 Run-level code evaluator (`cites_docs`)

Code evaluators run in LangSmith's sandbox: standard library plus numpy / pandas / jsonschema / scipy / scikit-learn, **no network**, one function `perform_eval(run, example)` returning `{feedback_key: score}`. Perfect for structural facts an LLM would only get approximately right: *did the answer cite a docs URL, and how long was it?*

The same source string is tested **locally** against a real run first — the fastest way to learn the exact shape of `run["outputs"]`.

In [ ]:
CITES_DOCS_CODE = r'''
import re

DOCS_URL = re.compile(r"https://(docs|reference)\.langchain\.com/\S+")

def _message(m):
    """Return (role, text) for a serialized LangChain message."""
    if not isinstance(m, dict):
        return "", ""
    if "kwargs" in m and isinstance(m.get("id"), list):          # {"lc":1, "id":[..,"AIMessage"], "kwargs":{...}}
        role, m = m["id"][-1].replace("Message", "").lower(), m["kwargs"]
    else:
        role = m.get("type") or m.get("role") or ""
    content = m.get("content", "")
    if not isinstance(content, str):
        content = "".join(b.get("text", "") for b in content if isinstance(b, dict))
    return role, content

def perform_eval(run, example=None):
    messages = (run.get("outputs") or {}).get("messages") or []
    answers = [text for role, text in map(_message, messages) if role in ("ai", "assistant") and text.strip()]
    answer = answers[-1] if answers else ""
    return {
        "cites_docs": int(bool(DOCS_URL.search(answer))),
        "answer_chars": len(answer),
    }
'''

# Test locally on the most recent root run before trusting it to the sandbox.
latest = next(client.list_runs(project_name=PROJECT_NAME, is_root=True, limit=1))
print("Local test on", latest.name, "->", run_code_evaluator_locally(CITES_DOCS_CODE, run_to_dict(latest)))

In [ ]:
cites_docs_id = await upsert_evaluator(
    client, "chat-langchain-cites-docs", type="code",
    code_evaluator={"code": CITES_DOCS_CODE, "language": "python"},
)
rule = attach_evaluator(
    client, cites_docs_id,
    project_name=PROJECT_NAME, display_name="cites-docs (code)", filter="eq(is_root, true)",
)
print("Evaluator:", cites_docs_id)
print("Rule:     ", rule["url"])

### 3.3 Thread-level (multi-turn) LLM judge (`conversation_resolved`)

Run-level judges can't see that the user *had to ask three times*. A thread-level evaluator waits until a thread has been idle for the project's **idle time**, stitches every turn into one conversation, and judges it once. Two things make it work:

- The prompt uses the special `{all_messages}` variable — LangSmith fills it with the assembled conversation.
- The rule sets `group_by="thread_id"`. Both agents emit `messages` at the top level of each trace, which is the format thread assembly needs.

> **Idle time is a project setting** (Tracing project → Settings), default 10 minutes, minimum 2. Set it to 2 for the workshop so the seeded threads get judged during the session. There is no API for it yet.

The rubric is deliberately "perceived error" shaped: did the user get what they came for without repeating, rephrasing, or correcting the assistant? (LangSmith ships a managed *Perceived Error (Tuned)* thread evaluator with the same intent — see the docs if you'd rather not maintain a prompt.)

In [ ]:
class ConversationGrade(BaseModel):
    """Grade for a whole multi-turn conversation."""
    conversation_resolved: bool = Field(description=(
        "True if, by the end of the thread, the user got what they were asking for without having to repeat, "
        "rephrase, or correct the assistant."
    ))
    comment: str = Field(description="One sentence: 'resolved', or which turn went wrong and why.")

THREAD_SYSTEM = (
    "You review a whole multi-turn conversation between a user and a LangChain documentation assistant. "
    "Judge the conversation as a whole, not a single turn: was the user's goal met, and did they have to repeat, "
    "rephrase, or correct the assistant to get there?"
)
THREAD_HUMAN = "Conversation:\n{all_messages}"

pushed = push_judge_prompt(
    client, "chat-langchain-conversation-resolved",
    system=THREAD_SYSTEM, human=THREAD_HUMAN, schema=ConversationGrade,
)
conversation_id = await upsert_evaluator(
    client, "chat-langchain-conversation-resolved", type="llm",
    llm_evaluator={
        "prompt_repo_handle": pushed["handle"],
        "commit_hash_or_tag": pushed["commit"],
        "variable_mapping": {"all_messages": "all_messages"},
    },
    model_configuration=JUDGE_MODEL_CONFIG,
)
rule = attach_evaluator(
    client, conversation_id,
    project_name=PROJECT_NAME, display_name="conversation-resolved (thread judge)",
    filter="", group_by="thread_id",
)
print("Evaluator:", conversation_id)
print("Rule:     ", rule["url"])

### 3.4 Chained code evaluator with extended stats (`needs_review`)

Evaluators can build on each other. With **include extended stats** the run passed to a code evaluator carries `feedback_stats` (plus `total_tokens`, `total_cost`), and a filter on `has(feedback_key, "answer_quality")` guarantees it only fires *after* the judge has scored the run. The result is one boolean, `needs_review`, that Part 4 routes on — a single signal is much easier to automate against than three.

In [ ]:
NEEDS_REVIEW_CODE = r'''
def perform_eval(run, example=None):
    stats = run.get("feedback_stats") or {}
    def avg(key):
        return (stats.get(key) or {}).get("avg")
    quality, cites = avg("answer_quality"), avg("cites_docs")
    flagged = (quality is not None and quality < 1) or (cites is not None and cites < 1)
    return {"needs_review": int(flagged)}
'''

needs_review_id = await upsert_evaluator(
    client, "chat-langchain-needs-review", type="code",
    code_evaluator={"code": NEEDS_REVIEW_CODE, "language": "python"},
)
rule = attach_evaluator(
    client, needs_review_id,
    project_name=PROJECT_NAME, display_name="needs-review (chained, extended stats)",
    filter='and(eq(is_root, true), has(feedback_key, "answer_quality"))',
    include_extended_stats=True,
)
print("Evaluator:", needs_review_id)
print("Rule:     ", rule["url"])

### 3.5 Trigger the evaluators and read the first numbers

Send a couple more conversations, wait for the evaluators to catch up (~30–90 s for run-level; thread-level after the idle time), then pull feedback back with `list_runs` and split it by `metadata.graph_id`. This is the first quantitative signal in the workflow-vs-deep-agent comparison.

In [ ]:
for graph_id in ("rag_workflow", "chat_deep_agent"):
    thread_ids[graph_id].append(run_conversation(graph_id, [
        "What's the difference between a LangGraph checkpointer and a store?",
        "Which one should hold user preferences?",
    ]))

print("\nWaiting for online evaluators...")
time.sleep(90)

since = datetime.now(timezone.utc) - timedelta(hours=2)
scored = list(client.list_runs(
    project_name=PROJECT_NAME, is_root=True, start_time=since,
    filter='eq(feedback_key, "answer_quality")', limit=100,
))

per_graph: dict[str, dict[str, list[float]]] = defaultdict(lambda: defaultdict(list))
for r in scored:
    graph_id = ((r.extra or {}).get("metadata") or {}).get("graph_id", "?")
    for key, stat in (r.feedback_stats or {}).items():
        if stat.get("avg") is not None:
            per_graph[graph_id][key].append(stat["avg"])

print(f"{len(scored)} scored root runs\n")
for graph_id, keys in per_graph.items():
    print(graph_id)
    for key, values in sorted(keys.items()):
        print(f"   {key:22} n={len(values):<3} mean={sum(values) / len(values):.2f}")

print("\nRules on the project:")
for r in list_run_rules(client, PROJECT_NAME):
    print(f"  - {r['display_name']}  (group_by={r.get('group_by')}, filter={r.get('filter') or '-'})")

---
# Part 4. Automation → annotation queue: the human checkpoint

Online evaluators are cheap and always on, but they are still a model grading a model. The **annotation queue** is where a person enters the loop, and its intent is narrow on purpose:

1. **Check the judge.** For each flagged run, is `answer_quality=false` fair? Record `human_correct` (Pass/Fail). Disagreements are how you later *align* the judge prompt.
2. **Write assertions, not answers.** When the run really is wrong, don't hand-write the perfect answer. Add short **assertions** — claims a correct answer *must* / *must not* satisfy (`must_cite_docs_url`, `must_mention_middleware_param`, `must_not_suggest_agentexecutor`). LangSmith saves them as `outputs.assertions` on a dataset example, with the run's inputs.
3. **Add to dataset.** "Add to Dataset & Next" drops the example into the queue's default dataset — the same one Part 5 evaluates against. Production failures become regression tests without anyone inventing golden text.

What reviewers should *not* do: rewrite the response, or grade style. The rubric below keeps them on task.

**Docs:** [Use assertions](https://docs.langchain.com/langsmith/assertions) · [Annotation queues via the SDK](https://docs.langchain.com/langsmith/annotation-queues-sdk) · [Automation rules](https://docs.langchain.com/langsmith/rules)

### 4.1 The dataset shell, the rubric, and the queue

Feedback configs are org-wide schemas; rubric items assign them to this queue with instructions. The dataset is created empty here so the queue can point at it — Part 5 seeds it.

In [ ]:
DATASET_NAME = "chat-langchain-assertions"
if client.has_dataset(dataset_name=DATASET_NAME):
    dataset = client.read_dataset(dataset_name=DATASET_NAME)
else:
    dataset = client.create_dataset(
        DATASET_NAME,
        description="Claim-based (assertion) references for Chat LangChain. Seeded by Module 6; grown from the annotation queue.",
    )

# Org-wide feedback schemas the rubric will reference (idempotent when identical).
client.create_feedback_config("human_correct", feedback_config={
    "type": "categorical", "categories": [{"value": 1, "label": "Pass"}, {"value": 0, "label": "Fail"}],
})
client.create_feedback_config("review_notes", feedback_config={"type": "freeform"})

# The shared assertion vocabulary. Assertion *keys* are the axis the offline experiment
# aggregates on, so they stay a small fixed set; the specific claim goes in the comment.
ASSERTION_KEYS = ["must_cite_source", "must_name_api", "must_explain_concept", "must_not_mislead"]

RUBRIC_INSTRUCTIONS = (
    "1) Was the online judge right? Mark human_correct Pass if the response was actually fine. "
    "2) If the response is wrong or unsupported, add ASSERTIONS describing what a correct answer must / must not "
    "include — do NOT rewrite the answer. Use one of these four keys and put the specific claim in the comment: "
    + ", ".join(ASSERTION_KEYS) + ". Repeat a key if the answer needs two claims of the same kind. Sticking to the "
    "shared vocabulary is what keeps the offline metrics comparable across examples. "
    "3) Click 'Add to Dataset & Next' so the assertions become an offline test case."
)
RUBRIC_ITEMS = [
    {"feedback_key": "human_correct", "description": "Was the judge's verdict on this run correct?",
     "value_descriptions": {"Pass": "Response was acceptable", "Fail": "Response really is wrong/unsupported"},
     "is_required": True},
    {"feedback_key": "review_notes", "description": "Anything the assertions don't capture.", "is_required": False},
]

QUEUE_NAME = "chat-langchain-needs-review"
existing = list(client.list_annotation_queues(name=QUEUE_NAME))
if existing:
    queue = existing[0]
    # create_annotation_queue leaves an existing queue alone, so push the rubric explicitly.
    client.update_annotation_queue(
        queue.id, rubric_instructions=RUBRIC_INSTRUCTIONS, rubric_items=RUBRIC_ITEMS)
else:
    queue = client.create_annotation_queue(
        name=QUEUE_NAME,
        description="Runs flagged by the needs_review evaluator (judge said no, or no docs URL cited).",
        rubric_instructions=RUBRIC_INSTRUCTIONS,
        rubric_items=RUBRIC_ITEMS,
    )
set_annotation_queue_default_dataset(client, queue.id, dataset.id)
print(f"Queue: {queue.name} ({queue.id}) -> default dataset {dataset.name}")

### 4.2 The automation rule

Same `create_run_rule` helper as Module 4, no evaluator this time — just a filter and `add_to_annotation_queue_id`. It fires on new **feedback**, so it runs a beat after `needs_review` lands. Automation rules also work at thread granularity (`group_by="thread_id"` → the whole conversation becomes one queue item); we route runs here because assertions are only available on run items.

In [ ]:
queue_rule = create_run_rule(
    client,
    project_name=PROJECT_NAME,
    display_name="route-needs-review",
    filter='and(eq(is_root, true), eq(feedback_key, "needs_review"), gt(feedback_score, 0.5))',
    add_to_annotation_queue_id=queue.id,
)
tenant_id = queue_rule["payload"]["tenant_id"]
QUEUE_URL = f"https://smith.langchain.com/o/{tenant_id}/annotation-queues/{queue.id}"
print("Rule:  ", queue_rule["url"])
print("Queue: ", QUEUE_URL)

# One-off, programmatic version of the same thing: push a whole seeded thread into the queue for review.
added = await client.annotation_queues.items.create(
    str(queue.id),
    items=[{"item_type": "THREAD", "thread_id": thread_ids["chat_deep_agent"][-1], "project_id": str(project.id)}],
)
print("Added thread item:", [(i.id, i.item_type) for i in (added.items or [])])

### 4.3 Reviewer walkthrough (in the UI)

Open the queue link above. For a flagged run:

1. Read the input and output. Fill in **human_correct** (was the judge right?).
2. Below **Feedback**, in **Assertions**, click **+ Add** and write a key + one-sentence claim. Repeat per claim. The **Outputs** panel switches to a read-only preview of your assertions — that preview, not the run's actual output, is what gets saved.
3. **Add to Dataset & Next** (⌘/Ctrl + Enter).

The resulting example in `chat-langchain-assertions` has the run's inputs and this `outputs` shape:

```json
{
  "assertions": [
    {"key": "must_cite_source", "comment": "The answer cites a docs.langchain.com or reference.langchain.com URL."},
    {"key": "must_name_api",    "comment": "The answer shows middleware being passed via create_agent(..., middleware=[...])."}
  ]
}
```

Part 5 seeds the same dataset with the same shape from code, so examples written by reviewers and examples written by engineers are indistinguishable to the evaluator.

---
# Part 5. Offline evaluation with claim-based assertions

Module 4's dataset had a `reference_answer` rubric per example. Here every reference is a **list of claims**. The evaluator scores every claim, then reports one score per claim **category** plus two aggregates, so a failing experiment tells you which *kind* of claim broke — and the feedback comment names the exact claim. Keys come from a small shared vocabulary (`ASSERTION_KEYS`) rather than one key per question, which is what makes the numbers comparable across examples and across experiments. Mechanical claims (`must_cite_*`) are checked by code; the rest by an LLM judge that is handed the claim text.

Inputs use the graph's own contract — `{"messages": [...]}` — so an example added from the annotation queue and an example seeded here look identical.

### 5.1 Seed the dataset

In [ ]:
# Assertions are a list of (key, comment) pairs, not a dict: an example may legitimately
# carry two claims of the same category, and the evaluator averages them.
def example(question: str, assertions: list[tuple[str, str]], *, expects_reference: bool = False) -> dict:
    return {
        "inputs": {"messages": [{"role": "user", "content": question}]},
        "outputs": {
            "assertions": [{"key": k, "comment": v} for k, v in assertions],
            "expects_reference": expects_reference,
        },
    }

CITE, API, CONCEPT, NO_MISLEAD = ASSERTION_KEYS          # the shared vocabulary from 4.1
CITE_CLAIM = (CITE, "Cites at least one docs.langchain.com or reference.langchain.com URL.")

seed_examples = [
    example("How do I add middleware to an agent built with create_agent?", [CITE_CLAIM,
        (API, "Shows middleware passed through create_agent(..., middleware=[...])."),
        (NO_MISLEAD, "Does not recommend the legacy AgentExecutor or initialize_agent.")]),
    example("What's the difference between LangChain and LangGraph, and when should I use each?", [CITE_CLAIM,
        (CONCEPT, "Explains that LangGraph is the lower-level orchestration runtime and create_agent is the higher-level abstraction built on it."),
        (CONCEPT, "Gives concrete guidance on when to pick one over the other.")]),
    example("How do I connect an agent to an MCP server over HTTP in Python?", [CITE_CLAIM,
        (API, "Mentions MultiServerMCPClient from langchain_mcp_adapters."),
        (API, "Shows a connection config with transport 'http' (or streamable_http) and a url.")]),
    example("What parameters does create_deep_agent accept for defining subagents?", [CITE_CLAIM,
        (API, "Lists the subagent dict fields: name, description, system_prompt, tools."),
        (NO_MISLEAD, "Does not invent parameters that do not exist on create_deep_agent.")], expects_reference=True),
    example("How do I make a LangGraph graph remember the conversation across turns?", [CITE_CLAIM,
        (API, "Explains compiling the graph with a checkpointer (e.g. MemorySaver / InMemorySaver)."),
        (API, "Explains passing a thread_id in config['configurable'].")]),
    example("What is the signature of init_chat_model and how do I pick the provider?", [CITE_CLAIM,
        (API, "Names the model string and the model_provider parameter (or the provider:model prefix form).")], expects_reference=True),
    example("How do I set up an online evaluator on a LangSmith tracing project?", [CITE_CLAIM,
        (CONCEPT, "Describes adding an evaluator to a tracing project (Evaluators tab / rule) with a filter and sampling rate."),
        (NO_MISLEAD, "Does not describe client.evaluate over a dataset as the online mechanism.")]),
    example("How do I pause a LangGraph graph for human approval before a tool runs, and resume it?", [CITE_CLAIM,
        (API, "Mentions interrupt() (or interrupt_before / interrupt_on) as the pausing mechanism."),
        (API, "Shows resuming with Command(resume=...).")]),
    example("What does the summarization middleware do in LangChain agents?", [CITE_CLAIM,
        (CONCEPT, "Explains that it condenses older conversation history when the context grows too large.")]),
    example("How can I stream tokens from an agent deployed on LangSmith using the LangGraph SDK?", [CITE_CLAIM,
        (API, "Mentions client.runs.stream (or the /runs/stream endpoint) and a stream_mode such as 'messages' or 'values'.")], expects_reference=True),
]

# Upsert on the question: re-running with edited claims updates the existing examples
# instead of silently skipping them (create_examples alone would).
by_inputs = {json.dumps(e.inputs, sort_keys=True): e.id for e in client.list_examples(dataset_id=dataset.id)}
to_create, to_update = [], []
for e in seed_examples:
    fingerprint = json.dumps(e["inputs"], sort_keys=True)
    if fingerprint in by_inputs:
        to_update.append({"id": by_inputs[fingerprint], "outputs": e["outputs"]})
    else:
        to_create.append(e)

if to_create:
    client.create_examples(dataset_id=dataset.id, examples=to_create)
if to_update:
    client.update_examples(dataset_id=dataset.id, updates=to_update)
print(f"Dataset '{DATASET_NAME}': {len(to_create)} created, {len(to_update)} updated")
print("Assertion keys:", ", ".join(ASSERTION_KEYS))
print("View:", dataset.url)


### 5.2 Targets and evaluators

Two targets, one per agent, returning the same shape (`answer`, `tool_calls`, `sources`). Two evaluators:

- `grade_assertions` — scores every claim (`must_cite_*` by regex, the rest by an LLM judge handed the claim text), then reports a **fixed** key set: `assertions_passed` (fraction satisfied — partial credit), `assertions_all_passed` (strict), and one score per assertion category. Failing claims are named in the comment, so you keep per-claim diagnosis without a column per question.
- `tool_usage` — `num_tool_calls`, and `used_reference_when_expected` for the API-shaped questions (the Deep Agent reaches the reference server through its `task` delegation, so that counts).

In [ ]:
DOCS_URL = re.compile(r"https://(docs|reference)\.langchain\.com/\S+")

# `aevaluate` hands the same `inputs` dict to the target and then to the evaluators, and a
# graph's state coercion can rewrite the message list it is given *in place* (the Deep Agent
# turns the dicts into Message objects). Copy, so the example's inputs survive for the evaluators.
def graph_input(inputs: dict) -> dict:
    return {"messages": [dict(m) for m in inputs["messages"]]}

def latest_question(inputs: dict) -> str:
    """Last user message, whether the example holds a dict or a coerced Message object."""
    last = inputs["messages"][-1]
    content = last["content"] if isinstance(last, dict) else last.content
    return content if isinstance(content, str) else "".join(
        b.get("text", "") for b in content if isinstance(b, dict))


async def run_rag(inputs: dict) -> dict:
    result = await rag.ainvoke(graph_input(inputs), config={"configurable": {"thread_id": str(uuid7())}})
    return {"answer": result["messages"][-1].text, "tool_calls": result["tool_calls"], "sources": result["sources"]}

async def run_deep(inputs: dict) -> dict:
    result = await deep.ainvoke(graph_input(inputs), config={"configurable": {"thread_id": str(uuid7())}})
    answer = result["messages"][-1].text
    return {"answer": answer, "tool_calls": extract_tool_calls(result["messages"]), "sources": DOCS_URL.findall(answer)}


class AssertionGrade(BaseModel):
    satisfied: bool = Field(description="True if the answer satisfies the claim.")
    reasoning: str = Field(description="One short sentence.")

ASSERTION_JUDGE_SYSTEM = (
    "You check whether an assistant's answer satisfies ONE claim about what a correct answer must or must not contain. "
    "Judge the claim literally and only from the answer text. Wording may differ from the claim."
)
assertion_judge = model.with_structured_output(AssertionGrade)

async def grade_assertions(inputs: dict, outputs: dict, reference_outputs: dict) -> list[dict]:
    """Score every claim, but report a fixed set of feedback keys.

    The claim text is what gets graded; the key is only the aggregation axis. So the
    experiment shows the same handful of metrics no matter which claims an example
    carries, and a category repeated within one example is averaged.
    """
    question = latest_question(inputs)
    results = []
    for assertion in reference_outputs["assertions"]:
        key, claim = assertion["key"], assertion["comment"]
        if key.startswith("must_cite"):
            score, why = int(bool(DOCS_URL.search(outputs["answer"]))), "regex check for a docs/reference URL"
        else:
            grade = await assertion_judge.ainvoke([
                SystemMessage(content=ASSERTION_JUDGE_SYSTEM),
                HumanMessage(content=f"Question: {question}\n\nAnswer:\n{outputs['answer']}\n\nClaim: {claim}"),
            ])
            score, why = int(grade.satisfied), grade.reasoning
        results.append({"key": key, "score": score, "comment": why})
    if not results:
        return []

    passed = sum(r["score"] for r in results)
    failed = [r for r in results if not r["score"]]
    detail = f"{passed}/{len(results)} claims passed"
    if failed:
        detail += " | failed: " + "; ".join(f"{r['key']} ({r['comment']})" for r in failed)

    feedback = [
        {"key": "assertions_passed",     "score": passed / len(results),       "comment": detail},
        {"key": "assertions_all_passed", "score": int(passed == len(results)), "comment": detail},
    ]
    per_category: dict[str, list[int]] = defaultdict(list)
    for r in results:
        per_category[r["key"]].append(r["score"])
    feedback += [{"key": k, "score": sum(v) / len(v)} for k, v in per_category.items()]
    return feedback


def tool_usage(outputs: dict, reference_outputs: dict) -> list[dict]:
    calls = outputs["tool_calls"]
    reached_reference = any(c in ("search_api", "get_symbol", "task") for c in calls)
    return [
        {"key": "num_tool_calls", "score": len(calls)},
        {"key": "used_reference_when_expected",
         "score": int(reached_reference) if reference_outputs.get("expects_reference") else 1},
    ]

### 5.3 Run both experiments

In [ ]:
rag_results = await aevaluate(
    run_rag, data=DATASET_NAME, evaluators=[grade_assertions, tool_usage],
    experiment_prefix="rag-workflow", max_concurrency=2, client=client,
)
deep_results = await aevaluate(
    run_deep, data=DATASET_NAME, evaluators=[grade_assertions, tool_usage],
    experiment_prefix="deep-agent", max_concurrency=2, client=client,
)
print("Experiments:", rag_results.experiment_name, "|", deep_results.experiment_name)

### 5.4 Pairwise: which answer would you rather get?

Per-example scores tell you *whether* each agent cleared the bar. A **pairwise** evaluator answers a different question — given the same input, which output is better? — which is exactly the workflow-vs-deep-agent question. `evaluate` takes the two existing experiments; `randomize_order=True` guards against position bias. The feedback key is prefixed `ranked_` by convention.

In [ ]:
class Preference(BaseModel):
    preferred: Literal["A", "B", "tie"]
    reasoning: str = Field(description="One sentence.")

PAIRWISE_SYSTEM = (
    "You compare two answers to the same LangChain documentation question. Prefer the answer that is more correct, "
    "more specific, and better cited (docs.langchain.com / reference.langchain.com URLs). Do not reward length. "
    "Answer 'tie' only if they are genuinely equivalent."
)
pairwise_judge = model.with_structured_output(Preference)

def ranked_preference(inputs: dict, outputs: list[dict]) -> list[int]:
    question = latest_question(inputs)
    grade = pairwise_judge.invoke([
        SystemMessage(content=PAIRWISE_SYSTEM),
        HumanMessage(content=(
            f"Question: {question}\n\n[Answer A]\n{outputs[0].get('answer', 'N/A')}\n\n[Answer B]\n{outputs[1].get('answer', 'N/A')}"
        )),
    ])
    return {"A": [1, 0], "B": [0, 1]}.get(grade.preferred, [0, 0])

pairwise = evaluate(
    (rag_results.experiment_name, deep_results.experiment_name),
    evaluators=[ranked_preference],
    randomize_order=True,
    experiment_prefix="pairwise-rag-vs-deep",
    max_concurrency=2,
    client=client,
)

### 5.5 Cost, latency, and scores side by side

Every experiment is a tracing project, so `read_project(include_stats=True)` gives run count, latency percentiles, tokens, cost, and per-key feedback averages for free.

In [ ]:
def experiment_stats(name: str, *extra_keys: str) -> dict:
    """Run count, latency, cost, and per-key feedback averages for one experiment.

    Feedback averages are computed from the feedback *rows*, not from
    `read_project(...).feedback_stats`: that aggregate is built asynchronously and is
    still partial for up to a minute after an experiment finishes, so reading it in the
    same cell that ran the experiment silently reports a fraction of the scores.
    Latency / tokens / cost below come from the project and lag the same way — give them
    a moment and re-run this cell if they look low.
    """
    p = client.read_project(project_name=name, include_stats=True)
    runs = list(client.list_runs(project_name=name, is_root=True))
    scores: dict[str, list[float]] = defaultdict(list)
    for f in client.list_feedback(run_ids=[r.id for r in runs]):
        if f.score is not None:
            scores[f.key].append(f.score)
    stats = {
        "runs": len(runs),
        "p50 latency (s)": round(p.latency_p50.total_seconds(), 1) if p.latency_p50 else None,
        "total tokens": p.total_tokens,
        "cost ($)": round(float(p.total_cost or 0), 4),
    }
    for key in ("assertions_passed", "assertions_all_passed", *ASSERTION_KEYS,
                "used_reference_when_expected", "num_tool_calls", *extra_keys):
        values = scores.get(key)
        stats[key] = round(sum(values) / len(values), 2) if values else None
    return stats

columns = {"rag_workflow": experiment_stats(rag_results.experiment_name),
           "chat_deep_agent": experiment_stats(deep_results.experiment_name)}

print(f"{'metric':30} {'rag_workflow':>16} {'chat_deep_agent':>16}")
for metric in next(iter(columns.values())):
    print(f"{metric:30} {str(columns['rag_workflow'][metric]):>16} {str(columns['chat_deep_agent'][metric]):>16}")

print("\nPairwise preference (ranked_preference, share of wins):")
for name in (rag_results.experiment_name, deep_results.experiment_name):
    fb = client.read_project(project_name=name, include_stats=True).feedback_stats or {}
    print(f"  {name:45} {fb.get('ranked_preference', {}).get('avg')}")
print("\nDataset with all experiments:", dataset.url)

### 5.6 Change one thing, re-measure

The table above is a verdict on two architectures, not the end of the loop. The point of the
ADLC is that a verdict becomes the next change: §5.5 says the Deep Agent is the weaker one at
naming exact APIs, so form a hypothesis, change **one** thing, and re-run the same dataset.

The hypothesis here: the Deep Agent under-delegates to its `api-reference` subagent because the
model isn't planning enough. So the knob is the model — and the metric that tests it is new:
`delegated`, whether the agent called `task` at all. **Measure the thing you changed**, or you
are back to reading tea leaves in `num_tool_calls`.

Three details that matter more than they look:

- `make_model("<name>")` builds the variant on the **same route** as `utils/models.py` — same
  provider, same gateway, same credentials — so a model swap doesn't quietly become a gateway
  bypass. Change the route in that file, not at the call site.
- The baseline is re-run **inside the same sweep**. A new metric doesn't exist on last run's
  experiment, so re-running the baseline is what makes the comparison apples-to-apples.
- `metadata={"variant": ...}` is what lets you group and filter these experiments in the UI
  later, rather than squinting at experiment names.

Swap `system_prompt` instead of `model` and the identical cell tests a prompt change — which is
the more common ADLC iteration.

> `num_tool_calls` counts only top-level calls, so the subagent's `search_api` / `get_symbol`
> work is invisible to it. Both variants undercount equally, so the comparison holds, but the
> absolute number understates the Deep Agent.


In [ ]:
from utils.models import make_model

# One knob, several settings. `{}` means "whatever utils/models.py exports".
DEEP_VARIANTS = {
    "baseline": {},
    "gpt-5.6-terra": {"model": make_model("gpt-5.6-terra")},
    # A prompt variant uses the identical shape:
    # "explicit-delegation": {"system_prompt": SYSTEM_PROMPT + (
    #     "\n- Before naming any class, function, or parameter in an answer, confirm it "
    #     "with the api-reference subagent via task().")},
}

# The metric that tests the hypothesis: did the agent delegate to the subagent at all?
def delegation(outputs: dict) -> list[dict]:
    return [{"key": "delegated", "score": int("task" in outputs["tool_calls"])}]

variant_experiments = {}
for label, overrides in DEEP_VARIANTS.items():
    agent = await make_local_agent(**overrides)

    # `_agent=agent` binds *this* iteration's agent. Without it, Python's late binding
    # would make every experiment run the last variant and the comparison a silent tie.
    async def run_variant(inputs: dict, _agent=agent) -> dict:
        result = await _agent.ainvoke(graph_input(inputs), config={"configurable": {"thread_id": str(uuid7())}})
        answer = result["messages"][-1].text
        return {"answer": answer, "tool_calls": extract_tool_calls(result["messages"]),
                "sources": DOCS_URL.findall(answer)}

    results = await aevaluate(
        run_variant, data=DATASET_NAME,
        evaluators=[grade_assertions, tool_usage, delegation],
        experiment_prefix=f"deep-{label}", max_concurrency=2, client=client,
        metadata={"agent": "chat_deep_agent", "variant": label},
    )
    variant_experiments[label] = results.experiment_name
    print(f"{label:16} -> {results.experiment_name}")

cols = {label: experiment_stats(name, "delegated") for label, name in variant_experiments.items()}
print(f"\n{'metric':30}" + "".join(f"{label:>18}" for label in cols))
for metric in next(iter(cols.values())):
    print(f"{metric:30}" + "".join(f"{str(cols[label][metric]):>18}" for label in cols))
print("\nCompare side by side:", dataset.url)


---
# Part 6. Insights — what are people actually asking, and where does it go wrong?

Evaluators answer questions you already thought to ask. **Insights** reads a sample of traces, summarizes each one with a prompt you control, extracts attributes, and clusters them into categories — surfacing the questions you *didn't* think to ask. The UI drives it from **+ New → New Insights Report**; we do the same from code so it can be re-run and scheduled.

Three knobs matter:

- The **summary prompt** decides what the clustering can see. Ours passes the whole thread (`{{all_thread_messages}}`) plus the evaluator feedback (`{{run.feedback}}`), so categories can reflect *outcomes*, not just topics.
- **Models**: a *Thinking* model clusters, a *Summarization* model summarizes. `INSIGHTS_THINKING_CONFIG` / `INSIGHTS_SUMMARY_CONFIG` from §3.0 pin them to named configurations; `None` uses the provider's defaults.
- **Attributes** are extracted per trace, steer the clustering, and show up as aggregates per category — e.g. the share of conversations in a category where the user had to repeat themselves.

> Requires a Plus/Enterprise plan and an Insights model configured under Settings → Model configurations. Reports take up to ~30 minutes; presenters, start one before the session.

**Docs:** [Insights](https://docs.langchain.com/langsmith/insights)

In [ ]:
INSIGHTS_ATTRIBUTES = {
    "question_type": {"type": "string",
        "description": "One of: how-to, concept, api-signature, troubleshooting, other."},
    "answered_with_citation": {"type": "boolean",
        "description": "True if the assistant's final answer includes a docs.langchain.com or reference.langchain.com URL."},
    "user_had_to_repeat": {"type": "boolean",
        "description": "True if the user rephrased, repeated, or corrected the assistant at any point in the conversation."},
}

report = create_insights_report(
    client, PROJECT_NAME,
    name=f"chat-langchain {datetime.now():%Y-%m-%d %H:%M}",
    attribute_schemas=INSIGHTS_ATTRIBUTES,
    last_n_hours=24,
    sample=200,
    thinking_model_config=INSIGHTS_THINKING_CONFIG,
    summarization_model_config=INSIGHTS_SUMMARY_CONFIG,
    # To compare the agents separately, run twice with a metadata filter, e.g.:
    # filter='and(eq(metadata_key, "graph_id"), eq(metadata_value, "chat_deep_agent"))',
)
print("Report:", report.link)

### 6.1 Schedule it

A saved config with a cron turns a one-off report into a recurring one. Time windows are recomputed on each run, and once a report has discovered top-level categories they are written back into the config, so week-over-week reports stay comparable.

In [ ]:
saved = save_insights_config(
    client, PROJECT_NAME,
    name="chat-langchain weekly",
    description="Weekly topics + failure modes for both docs agents.",
    schedule_cron="0 8 * * 1",          # Mondays 08:00 UTC
    attribute_schemas=INSIGHTS_ATTRIBUTES,
    last_n_hours=24 * 7,
    sample=500,
    thinking_model_config=INSIGHTS_THINKING_CONFIG,
    summarization_model_config=INSIGHTS_SUMMARY_CONFIG,
)
print("Saved config:", saved.get("id"), "| schedule:", saved.get("schedule_cron"))

In [ ]:
# Optional: block until the report finishes (up to ~30 min), then print the executive summary.
POLL_INSIGHTS = False
if POLL_INSIGHTS:
    finished = client.poll_insights(report=report)
    print_insights_summary(client.get_insights_report(report=finished, include_runs=False))

---
## Recap

| ADLC step | What we did | API |
|---|---|---|
| **Build** | Two agents on the docs MCP servers | `MultiServerMCPClient.get_tools()`, `StateGraph`, `create_deep_agent` |
| **Deploy** | One deployment, two graphs, a packaged `/chat` page | `langgraph.chat_langchain.json` (`graphs`, `http.app`), `langgraph deploy --config ...` |
| **Observe** | Multi-turn threads via the SDK, Studio, or the UI | `sdk.threads.create()`, `sdk.runs.wait(...)` |
| **Online evals** | LLM judge, code evaluator, thread judge, chained evaluator | `client.evaluators.create/update/list`, `attach_evaluator(..., group_by="thread_id", include_extended_stats=True)` |
| **Human review** | Flagged runs → queue → assertions → dataset | `create_feedback_config`, `create_annotation_queue(rubric_items=...)`, `create_run_rule(add_to_annotation_queue_id=...)` |
| **Offline evals** | Per-claim grading, tool usage, pairwise, cost table | `aevaluate`, `evaluate((exp_a, exp_b), randomize_order=True)`, `read_project(include_stats=True)` |
| **Insights** | Report now + weekly schedule | `POST /sessions/{id}/insights`, `.../insights/configs` |
| **Model configs** | Judges + Insights on named configurations | `get_model_configuration(name)` → `playground_settings_id`, `cluster_model`, `summary_model` |

### Workflow vs Deep Agent — what the numbers usually say

| | LangGraph workflow | Deep Agent |
|---|---|---|
| Predictability | Fixed steps, bounded cost, easy to unit-test per node | Variable trajectories; needs trajectory + thread evals |
| Latency / cost | Low, flat | Higher, spiky (subagent hops, page reads) |
| Precise API questions | Weak unless the route picks `reference` | Strong — drills into `get_symbol` |
| Vague / multi-hop questions | Fails politely ("not in the context") | Re-searches, reads pages, usually recovers |
| When to pick it | Known question shapes, tight SLAs, regulated flows | Open-ended support, exploration, "just figure it out" |

The honest answer for a docs assistant is often **both**: the workflow as the default path and the Deep Agent behind a router for the questions the workflow can't ground — and the same evaluators, queue, and Insights report keep both honest.

**Next:** Module 5 — Engine watches this same project and turns recurring failures into issues, PRs, and evaluators automatically.

In [ ]:
# Teardown (uncomment to remove everything this notebook created).
# for r in list_run_rules(client, PROJECT_NAME):
#     delete_run_rule(client, r["id"])
# for evaluator_id in (answer_quality_id, cites_docs_id, conversation_id, needs_review_id):
#     await client.evaluators.delete(evaluator_id, delete_run_rules=True)
# client.delete_dataset(dataset_id=dataset.id)